#📊 **KPI Engineering & Feature Creation**

####Objective:
This step is focused on unlocking deeper insights from the foundational dataset. Instead of relying solely on absolute figures, we engineer Key Performance Indicators (KPIs) that capture proportional strategic strength — for example, evaluating defense expenditure in relation to a nation’s total GDP.

By transforming raw metrics into meaningful ratios and performance indicators, the analysis becomes more context-driven and strategically relevant. In parallel, essential metadata such as continent and regional classifications is integrated into the dataset. This enrichment enables powerful filtering, segmentation, and aggregation capabilities, forming the backbone for dynamic and interactive dashboard visualizations.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- CONFIGURATION ---
INPUT_FILE = "military_cleaned.csv"
OUTPUT_FILE = "military_final.csv"

# --- STEP 1: LOAD CLEAN DATASET ---
print(f"Step 1: Loading '{INPUT_FILE}'...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"   -> Loaded {len(df)} rows.")
except FileNotFoundError:
    raise Exception(f"ERROR: Could not find '{INPUT_FILE}'. Please complete Task 6 first.")

# Strip whitespace from columns to prevent KeyErrors
df.columns = df.columns.str.strip().str.lower()

# --- STEP 2: METADATA ENRICHMENT ---
print("\nStep 2: Adding Metadata (Continent, Region, Alliance)...")

nato_list = [
    "albania", "belgium", "bulgaria", "canada", "croatia", "czechia",
    "denmark", "estonia", "finland", "france", "germany", "greece",
    "hungary", "iceland", "italy", "latvia", "lithuania", "luxembourg",
    "montenegro", "netherlands", "north-macedonia", "norway", "poland",
    "portugal", "romania", "slovakia", "slovenia", "spain", "sweden",
    "turkiye", "united-kingdom", "united-states-of-america", "united-states"
]

df['alliance_flag'] = (
    df['country']
    .str.lower()
    .str.strip()
    .str.replace(' ', '-', regex=False)
    .apply(lambda x: 'NATO' if x in nato_list else 'Non-Aligned')
)

def get_geo_details(country):
    c = str(country).lower().strip().replace(' ', '-')

    if any(x in c for x in ['united-states', 'canada']): return 'Americas', 'Northern America'
    if any(x in c for x in ['chile', 'brazil', 'argentina', 'colombia', 'peru', 'ecuador', 'venezuela', 'uruguay', 'paraguay', 'bolivia', 'suriname']): return 'Americas', 'South America'
    if any(x in c for x in ['mexico', 'cuba', 'panama', 'guatemala', 'belize', 'dominican-republic', 'el-salvador', 'honduras', 'nicaragua']): return 'Americas', 'Central America'

    if any(x in c for x in ['india', 'pakistan', 'afghanistan', 'bangladesh', 'sri-lanka', 'nepal', 'bhutan']): return 'Asia', 'Southern Asia'
    if any(x in c for x in ['china', 'japan', 'korea', 'taiwan', 'mongolia']): return 'Asia', 'Eastern Asia'
    if any(x in c for x in ['vietnam', 'thailand', 'indonesia', 'philippines', 'singapore', 'malaysia', 'cambodia', 'laos', 'myanmar']): return 'Asia', 'South-East Asia'
    if any(x in c for x in ['kazakhstan', 'uzbekistan', 'turkmenistan', 'kyrgyzstan', 'tajikistan']): return 'Asia', 'Central Asia'

    if any(x in c for x in ['russia', 'ukraine', 'poland', 'romania', 'belarus', 'czechia', 'hungary', 'bulgaria', 'moldova', 'albania']): return 'Europe', 'Eastern Europe'
    if any(x in c for x in ['france', 'germany', 'belgium', 'netherlands', 'switzerland', 'austria', 'luxembourg']): return 'Europe', 'Western Europe'
    if any(x in c for x in ['united-kingdom', 'sweden', 'norway', 'denmark', 'finland', 'ireland', 'iceland', 'estonia', 'latvia', 'lithuania']): return 'Europe', 'Northern Europe'
    if any(x in c for x in ['italy', 'spain', 'greece', 'portugal', 'serbia', 'croatia', 'slovenia', 'bosnia', 'montenegro', 'north-macedonia', 'kosovo']): return 'Europe', 'Southern Europe'

    if any(x in c for x in ['israel', 'saudi', 'turkiye', 'iran', 'iraq', 'uae', 'qatar', 'jordan', 'kuwait', 'lebanon', 'oman', 'yemen', 'bahrain', 'syria', 'cyprus', 'georgia', 'armenia', 'azerbaijan']): return 'Middle East', 'Western Asia (Middle East)'

    if any(x in c for x in ['egypt', 'algeria', 'morocco', 'libya', 'tunisia']): return 'Africa', 'Northern Africa'
    if any(x in c for x in ['nigeria', 'south-africa', 'kenya', 'ethiopia', 'angola', 'benin', 'botswana', 'burundi', 'cameroon', 'central-african-republic', 'chad', 'congo', 'djibouti', 'eritrea', 'gabon', 'gambia', 'ghana', 'guinea', 'ivory-coast', 'lesotho', 'liberia', 'madagascar', 'malawi', 'mali', 'mauritania', 'mauritius', 'mozambique', 'namibia', 'niger', 'senegal', 'sierra-leone', 'somalia', 'sudan', 'tanzania', 'uganda', 'zambia', 'zimbabwe']): return 'Africa', 'Sub-Saharan Africa'

    if any(x in c for x in ['australia', 'new-zealand']): return 'Oceania', 'Australia & New Zealand'

    return 'Other', 'Other Subregion'

# Apply Metadata
df[['continent', 'region']] = df['country'].apply(lambda x: pd.Series(get_geo_details(x)))
print("   -> 'continent', 'region', and 'alliance_flag' columns successfully mapped.")

# --- STEP 3: KPI CALCULATIONS ---
print("\nStep 3: Calculating KPIs...")

# Ensure numeric types for asset columns
num_cols = ['total_aircraft', 'tanks', 'naval_assets', 'total_manpower', 'defense_budget', 'gdp']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Extract power_index safely (filling missing values with 999 to drop them to the bottom of rankings)
if 'power_index' in df.columns:
    df['power_index'] = pd.to_numeric(df['power_index'].astype(str).str.extract(r'([\d.]+)')[0], errors='coerce').fillna(999)
else:
    df['power_index'] = 999

# Combine Military Hardware
df['total_military_hardware'] = df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0)

# KPI 1: Assets per Capita
df['assets_per_capita'] = (df['total_military_hardware'] / df.get('total_manpower', 1).replace(0, 1)).round(7)

# KPI 2: Budget-to-GDP Ratio
df['budget_gdp_ratio'] = (df.get('defense_budget', 0) / df.get('gdp', 1).replace(0, 1)).round(5)

# KPI 3: Power Index Rank & Gap
df['rank'] = df['power_index'].rank(ascending=True, method='min').astype(int)
top_rank_value = df['rank'].min()
df['power_index_rank_gap'] = df['rank'] - top_rank_value

print("   -> KPIs successfully calculated.")

# --- STEP 5: VALIDATE KPI VALUES ---
print("\nStep 5: Validating Data...")

if np.isinf(df['budget_gdp_ratio']).any():
    print("   WARNING: Infinite values found in Budget-to-GDP. Replacing with 0.")
    df.replace([np.inf, -np.inf], 0, inplace=True)

sample = df[df['country'] == 'India'][['country', 'assets_per_capita', 'budget_gdp_ratio', 'rank']]
if not sample.empty:
    print(f"   -> Validation Sample (India):\n{sample.to_string(index=False)}")
else:
    print("   -> Validation Sample (India) not found.")

# --- STEP 6: EXPORT FINAL DATASET ---
print(f"\nStep 6: Exporting to '{OUTPUT_FILE}'...")
df.to_csv(OUTPUT_FILE, index=False)

print(f"[SUCCESS] Final analytics file ready: {OUTPUT_FILE}")

**Takeaway:** Enriched the dataset by integrating 3 geographic dimensions and engineering 3 strategic KPIs — `assets_per_capita`, `budget_gdp_ratio`, and `power_index_rank_gap`. These enhancements were successfully computed across all 145 records and finalized in the exported dataset `military_final.csv`, making it fully analysis-ready.